## Deliverable 2

Imports 

In [19]:

import re
from pathlib import Path
import json
from typing import Any
import pandas as pd
import numpy as np


Task 1 -  File ingestion and regular expressions

In [ ]:

def read_file(filepath): #Leser en fil og returnerer (tekstinnhold, koding som ble brukt)
    path = Path(filepath)
    raw = path.read_bytes()

    try:
        # UTF-8 er streng og feiler hvis bytene ikke er gyldig UTF-8
        text = raw.decode("utf-8")
        encoding = "utf-8"
    except UnicodeDecodeError:
        # Latin-1 kan dekode alle bytes, så den brukes som reserve
        text = raw.decode("latin-1")
        encoding = "latin-1"

    return text, encoding



LOG_PATTERN = re.compile(
    r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] "
    r"\[(?P<register_id>\w+)\] "
    r"item=(?P<item>.+?) "   
    r"price=(?P<price>\d+(?:\.\d+)?) "
    r"(?P<currency>[A-Z]{3}) "
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)"
)


def parse_log(text):
    """Returnerer en liste med dictionaries, én per gyldig salgslinje."""
    records = []

    for line in text.splitlines():
        match = LOG_PATTERN.fullmatch(line.strip())
        if match is None:
            continue  # støylinje, tom linje eller ugyldig linje

        record = match.groupdict()
        record["price"] = float(record["price"])
        records.append(record)

    return records


def parse_log_file(filepath):
    text, _encoding = read_file(filepath)
    return parse_log(text)


if __name__ == "__main__":
    eksempel = (
        "--- PRINTER PAPER LOW WARNING ---\n"
        "[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)\n"
        "=== NETWORK SYNC OK ===\n"
        "[2026-10-15 08:31:12] [POS_02] item=Iced Latte price=55 NOK (REFUND)\n"
        "[2026-10-15 08:32:00] [POS_01] item=Mocha price=5.00 USD (PENDING)\n"
    )
    for rec in parse_log(eksempel):
        print(rec)

{'timestamp': '2026-10-15 08:30:00', 'register_id': 'POS_01', 'item': 'Latte', 'price': 4.5, 'currency': 'EUR', 'status': 'SUCCESS'}
{'timestamp': '2026-10-15 08:31:12', 'register_id': 'POS_02', 'item': 'Iced Latte', 'price': 55.0, 'currency': 'NOK', 'status': 'REFUND'}


Task 2 - Recursive utility functions

In [21]:

def find_files(directory: str | Path) -> list[Path]:
    """
    Recursively find all .log files in a directory tree.

    Args:
        directory: Root directory to search.

    Returns:
        A list of paths to .log files.
    """
    directory = Path(directory)
    log_files: list[Path] = []

    for item in directory.iterdir():
        if item.is_dir():
            log_files.extend(find_files(item))
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files


def find_active_registers(data: Any) -> set[str]:
    """
    Recursively search a nested JSON structure for active registers.

    A register is considered active if:
        - "active" is True
        - an "id" field is present

    Args:
        data: Nested JSON data consisting of dictionaries,
              lists, and primitive values.

    Returns:
        A set containing the IDs of all active registers.
    """
    active_ids: set[str] = set()

    if isinstance(data, dict):
        if data.get("active") is True and "id" in data:
            active_ids.add(data["id"])

        for value in data.values():
            active_ids |= find_active_registers(value)

    elif isinstance(data, list):
        for item in data:
            active_ids |= find_active_registers(item)

    return active_ids


def load_active_registers(
    config_path: str | Path = "data/store_config.json",
) -> set[str]:
    """
    Load store configuration and return active register IDs.

    Args:
        config_path: Path to the JSON configuration file.

    Returns:
        A set of active register IDs.
    """
    path = Path(config_path)

    with path.open("r", encoding="utf-8") as file:
        config = json.load(file)

    return find_active_registers(config)


if __name__ == "__main__":
    example = {
        "company": "Campus Coffee",
        "regions": [
            {
                "region_name": "Main Campus",
                "stores": [
                    {
                        "name": "Library Cafe",
                        "registers": [
                            {"id": "POS_01", "active": True},
                            {"id": "POS_02", "active": False},
                        ],
                    }
                ],
            }
        ],
    }

    print(find_active_registers(example))



{'POS_01'}


Task 3 - Functions for cleaning and analysing data using NumPy

In [22]:

def remove_price_outliers(prices: np.ndarray) -> np.ndarray:
    
    """
    Removes price outliers by using the 3-sigma rule
    Prices that are more then 3 standard derivation from the mean are replaced with the median price
    """
    
    prices = np.asarray(prices, dtype = float)
    
    mean = np.mean(prices)
    std = np.std(prices)
    
    #Masks that find outliners
    outliers_mask = np.abs(prices - mean) > 3 * std
    #Finds the valid prices
    valid_mask = ~outliers_mask 
    
    #Finds the median
    valid_prices = prices[valid_mask]
    median = np.median(valid_prices)
    
    #Replace abberation with median by the valid prices 
    cleaned_prices = prices.copy()
    cleaned_prices[outliers_mask] = median
    
    return cleaned_prices

Task 4 - Data analysis using Pandas

In [23]:
def calculate_refund_ratio(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate refund ratios for all items.
    Counts succesfull transactions and refunded transactions for each item
    """
    
    #Find numbers sucessfully and refund per hour
    sales = (df[df["status"] == "SUCCESS"]
             .groupby("item")
             .size()
             )
    
    refund = (df[df["status"] == "REFUND"]
              .groupby("item")
              .size()
              )

    #Link togheter as an doubble DataFrame
    counts = pd.DataFrame({
        "sales": sales,
        "refund": refund
        }).fillna(0)
    
    #Calculate ratio
    counts["refund_ratio"] = compute_refund_ratio(
        counts["sales"].to_numpy(),
        counts["refund"].to_numpy()
        )
    
    return counts

Task 5 - Create full workflow

In [25]:
def main() -> None:
    """
    Execute the complete data-processing workflow
    """
    
    log_files = find_files("data/2026")
    transactions = parse_log(log_files)
    df = create_dataframe(transactions)
    
    #Remove price-outliers
    df["price"] = remove_price_outliers(df["price"].to_numpy())
    
    #Rectify valuta 
    df.loc[df["currency"] == "NOK", "price"] *= 10.0
    
    
    #Rapports
    hourly = hourly_revenue(df)
    pivot = create_pivot_table(df)
    refunds = calculate_refund_ratio(df)
    
    #Save files
    df.to_csv("cleaned_transactions.csv", index = False)
    hourly.to_json("hourly_revenue.json")
    pivot.to_csv("status_pivot_table.csv")
    refunds.to_csv("item_refund_ratios.csv")